# Sentiment Analysis Model - IMDB Movie Reviews
**AI & ML Internship Track - Week 4 (Advanced / Capstone)**

This notebook builds a sentiment analysis model that classifies movie reviews as positive or negative, using TF-IDF for text feature extraction and a classification model.

**Dataset used:** IMDB Dataset of 50K Movie Reviews (from Kaggle). It has two columns: `review` (the text) and `sentiment` (positive/negative).

## 1. Install and Import Libraries
If needed, run this in your terminal first: `pip install scikit-learn nltk`

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

## 2. Load the Dataset
Make sure `IMDB Dataset.csv` is in the same folder as this notebook. If your file has a different name, change it below.

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

### Check the balance of positive vs negative reviews
It's important to know if the dataset is roughly balanced, since a very imbalanced dataset can make accuracy misleading.

In [ ]:
df['sentiment'].value_counts()

## 3. Clean the Text
We lowercase everything and remove punctuation/special characters so the model focuses on the actual words, not formatting noise. We also remove leftover HTML tags, which are common in this dataset (e.g. `<br />`).

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)          # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)    # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()   # remove extra whitespace
    return text

df['cleaned_text'] = df['review'].apply(clean_text)
df[['review', 'cleaned_text']].head()

## 4. Convert Text to Numeric Features with TF-IDF
Models can't read raw text, so we use **TF-IDF** (Term Frequency-Inverse Document Frequency), which turns each review into a vector of word-importance scores. We limit it to the 5,000 most informative words to keep things fast.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['sentiment']

print('Feature matrix shape:', X.shape)

## 5. Split into Training and Testing Sets
80% of the data trains the model, 20% is held back for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

## 6. Train a Classification Model
We use Logistic Regression, which works well for TF-IDF text features.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

## 7. Evaluate the Model
We check accuracy and F1-score, and print a confusion matrix to see exactly how many reviews were correctly/incorrectly classified as positive or negative.

In [ ]:
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, pos_label='positive')

print(f'Accuracy: {accuracy:.3f}')
print(f'F1-score: {f1:.3f}')

print('\nConfusion Matrix:')
print(confusion_matrix(y_test, predictions))

## 8. Compare with Naive Bayes
Multinomial Naive Bayes is another common baseline for text classification. We train it too, to see how it compares to Logistic Regression.

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)

nb_accuracy = accuracy_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions, pos_label='positive')

print(f'Naive Bayes Accuracy: {nb_accuracy:.3f}')
print(f'Naive Bayes F1-score: {nb_f1:.3f}')

## 9. Test on My Own Example Sentences
We write 3 original sentences, clean and vectorize them the same way as the training data, and see what the model predicts.

In [ ]:
my_sentences = [
    'This was the best experience ever, I loved every second of it.',
    'This was a complete waste of time, I want my money back.',
    'It was okay, not great but not terrible either.'
]

cleaned_sentences = [clean_text(s) for s in my_sentences]
vectorized_sentences = vectorizer.transform(cleaned_sentences)
my_predictions = model.predict(vectorized_sentences)

for sentence, prediction in zip(my_sentences, my_predictions):
    print(f'"{sentence}" -> {prediction}')

## 10. Final Summary
*(Replace the bracketed parts below with your own actual results after running the notebook.)*

**Pipeline:** Raw IMDB movie reviews were cleaned (lowercased, HTML tags and punctuation removed), then converted into numeric features using TF-IDF with the top 5,000 words. The data was split 80/20 into training and testing sets. A Logistic Regression model was trained on the TF-IDF features to classify reviews as positive or negative, and compared against a Multinomial Naive Bayes baseline.

**Results:** The Logistic Regression model achieved an accuracy of about **[X]%** and an F1-score of **[X]**, meaning it correctly classified the large majority of reviews. Naive Bayes performed [similarly / slightly worse / slightly better], which is typical for this kind of text classification task. On my own 3 test sentences, the model correctly identified [how many out of 3] as positive/negative.

**Limitation:** The model relies on individual word frequency (TF-IDF) and has no understanding of context, word order, negation, or sarcasm. For example, a sentence like 'not bad at all' can confuse it, since it contains the word 'bad' even though the overall sentiment is positive. A more advanced approach using word embeddings or a transformer-based model (like BERT) would likely handle these cases better.